In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

# Load your dataset

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/india_cities.csv')
df.head(10)

# Data Cleaning

In [ ]:
# Check missing values
df.isnull().sum()


Remove records without coordinates:

In [ ]:
df = df.dropna(subset=['Latitude', 'Longitude'])

# Make sure coordinates and numeric columns are numeric
df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Population'] = pd.to_numeric(df['Population'], errors='coerce')
df['Digital Elevation Model'] = pd.to_numeric(
    df['Digital Elevation Model'], errors='coerce'
)

# Fill remaining NaN
df['State'].fillna('Unknown', inplace=True)

df = df.set_index('Name')
df.head()

# Interactive Population Distribution of Indian Cities

In [ ]:
fig = px.scatter_geo(
    df.reset_index(),
    lat='Latitude',
    lon='Longitude',
    hover_name='Name',
    hover_data={
        'Population': True,
        'Digital Elevation Model': True,
        'Latitude': False,
        'Longitude': False
    },
    size='Population',
    size_max=40,
    color='Digital Elevation Model',
    color_continuous_scale='Viridis',
    title='Interactive Population Distribution of Indian Cities',
    center=dict(lat=22, lon=80),
    projection='equirectangular',
    scope='asia'
)
fig.update_geos(
    projection_scale=4,
    showocean=True, oceancolor='lightblue',
    showland=True,  landcolor='lightyellow',
    showcountries=True, countrycolor='gray'
)
fig.update_layout(height=600)
fig.show()

# Top 20 Most Populated Cities

In [ ]:
top20 = (
    df.reset_index()
    .sort_values(
        'Population',
        ascending=False
    )
    .head(20)
)

top20[
    [
        'Name',
        'Latitude',
        'Longitude',
        'Population'
    ]
]

Visualize them:

In [ ]:
plt.figure(figsize=(12, 8))

plt.barh(
    top20['Name'],
    top20['Population']
)

plt.xlabel('Population')
plt.ylabel('City')
plt.title('Top 20 Most Populated Indian Cities')

plt.gca().invert_yaxis()

plt.show()

# Top 20 Cities on a Geographic Map

In [ ]:
top20 = df.reset_index().nlargest(20, 'Population')

plt.figure(figsize=(12, 10))

plt.scatter(
    df.reset_index()['Longitude'],
    df.reset_index()['Latitude'],
    s=5,
    alpha=0.2
)

plt.scatter(
    top20['Longitude'],
    top20['Latitude'],
    s=top20['Population'] / 10000,
    alpha=0.8
)

for _, row in top20.iterrows():
    plt.text(
        row['Longitude'],
        row['Latitude'],
        row['Name'],
        fontsize=8
    )

plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.title(
    "Geographical Distribution of India's Most Populated Cities"
)

plt.show()

# GeoPandas Visualization

In [ ]:
gdf = gpd.GeoDataFrame(
    df.reset_index(),
    geometry=gpd.points_from_xy(
        df.reset_index()['Longitude'],
        df.reset_index()['Latitude']
    ),
    crs='EPSG:4326'
)

gdf.head()


Check the geometry:

In [ ]:
gdf.plot(
    figsize=(12, 10),
    markersize=5,
    alpha=0.5
)

plt.title('Indian Cities – GeoPandas Spatial Visualization')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.show()

# Elevation Map Using GeoPandas

In [ ]:
gdf.plot(
    column='Digital Elevation Model',
    figsize=(12, 10),
    legend=True,
    markersize=10,
    alpha=0.7,
    cmap='terrain'
)

plt.title('Spatial Distribution of Elevation Across Indian Cities')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.show()

## Satellite Basemap (Contextily + GeoPandas)

In [ ]:
try:
    import contextily as ctx

    gdf_web = gdf.to_crs(epsg=3857)   # Web Mercator for tile alignment

    fig, ax = plt.subplots(figsize=(14, 16))
    gdf_web.plot(
        ax=ax, column='Population',
        cmap='OrRd',
        markersize=gdf_web['Population'] / 80000,
        alpha=0.85, legend=True,
        legend_kwds={'label': 'Population', 'shrink': 0.5, 'format': '%,.0f'}
    )
    # Add real satellite imagery as basemap
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=5)
    ax.set_title(
        '🛰️  India Cities on Satellite Imagery (Bubble = Population)',
        fontsize=15, fontweight='bold', pad=12
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

except ImportError:
    # Dark-mode fallback
    fig, ax = plt.subplots(figsize=(10, 12))
    sc = ax.scatter(df.reset_index()['Longitude'], df.reset_index()['Latitude'],
                    c=df.reset_index()['Population'], cmap='OrRd',
                    s=df.reset_index()['Population']/80000, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Population')
    ax.set_facecolor('#1a2a4a')
    ax.set_title('India Cities – Fallback Dark Map', fontsize=14)
    plt.tight_layout(); plt.show()

## Interactive Map – Satellite Imagery + City Markers (Folium)
> Switch layers with the ≡ control.  Click any marker for details.

| Colour | City Tier |
|--------|-----------|
| 🔴 Red | Tier 1 Metro |
| 🟠 Orange | Tier 1 |
| 🔵 Blue | Tier 2 |
| 🟢 Green | Tier 3 |


In [ ]:
def tier_color(tier):
    return {
        'Tier 1 (Metro)': 'red',
        'Tier 1':         'orange',
        'Tier 2':         'blue',
        'Tier 3':         'green'
    }.get(str(tier), 'gray')

m_sat = folium.Map(location=[22.5, 80.0], zoom_start=5, tiles=None)

# Satellite imagery tile layer
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri WorldImagery',
    name='🛰️ Satellite',
    overlay=False, control=True
).add_to(m_sat)
folium.TileLayer('OpenStreetMap',       name='🗺️ Street Map').add_to(m_sat)
folium.TileLayer('CartoDB positron',    name='☁️ Light').add_to(m_sat)
folium.TileLayer('CartoDB dark_matter', name='🌑 Dark').add_to(m_sat)

for _, row in df.reset_index().iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=max(4, row['Population'] / 800_000),
        color=tier_color(row['City_Tier']),
        fill=True, fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>🏙️ {row['Name']}</b><br>"
            f"📍 State: {row['State']}<br>"
            f"👥 Population: <b>{row['Population']:,}</b><br>"
            f"⛰️ Elevation: {row['Digital Elevation Model']:.0f} m<br>"
            f"🏷️ Tier: {row['City_Tier']}",
            max_width=260
        ),
        tooltip=f"{row['Name']} · {row['City_Tier']} · Pop: {row['Population']:,}"
    ).add_to(m_sat)

folium.LayerControl().add_to(m_sat)
m_sat.save('india_satellite_map.html')
display(m_sat)

# 📊 Advanced Interactive Charts (Plotly)
> Hover · Zoom · Pan · Filter by Legend · Download as PNG


## Top 20 Cities – Interactive Bar (by Population)

In [ ]:
top20 = df.reset_index().nlargest(20, 'Population')

fig = px.bar(
    top20.sort_values('Population'),
    x='Population', y='Name',
    orientation='h',
    color='Region',
    text=top20.sort_values('Population')['Population'].apply(
        lambda x: f'{x/1e6:.2f}M'
    ),
    title='🏙️ Top 20 Most Populous Cities in India',
    labels={'Population': 'Population', 'Name': 'City'}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=600, xaxis_tickformat=',')
fig.show()

## Elevation vs Population – Interactive Scatter

In [ ]:
fig = px.scatter(
    df.reset_index(),
    x='Digital Elevation Model', y='Population',
    color='Region', size='Population',
    hover_name='Name',
    hover_data={'State': True, 'City_Tier': True,
                'Digital Elevation Model': True, 'Population': True},
    title='⛰️ City Elevation vs Population',
    labels={'Digital Elevation Model': 'Elevation (m)', 'Population': 'Population'}
)
fig.update_layout(height=550, yaxis_tickformat=',')
fig.show()

## Population by Region – Interactive Treemap

In [ ]:
fig = px.treemap(
    df.reset_index(),
    path=['Region', 'City_Tier', 'Name'],
    values='Population',
    color='Population',
    color_continuous_scale='RdYlGn',
    title='🌳 India Cities – Region → Tier → City (Area = Population)'
)
fig.update_layout(height=650)
fig.show()

## 3D Map – Lon × Lat × Elevation (colour = Population)

In [ ]:
fig = px.scatter_3d(
    df.reset_index(),
    x='Longitude', y='Latitude', z='Digital Elevation Model',
    color='Population', size='Population',
    hover_name='Name',
    color_continuous_scale='Hot',
    title='🌐 3D India City Map – Lon × Lat × Elevation',
    size_max=30
)
fig.update_layout(height=700, scene=dict(
    xaxis_title='Longitude (°E)',
    yaxis_title='Latitude (°N)',
    zaxis_title='Elevation (m)'
))
fig.show()

## 🔵 Cluster Map – Zoom In to Expand Clusters (Folium)

In [ ]:
from folium.plugins import MarkerCluster

m_cl = folium.Map(
    location=[22.5, 80.0], zoom_start=5,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

cluster = MarkerCluster(name='India Cities').add_to(m_cl)

for _, row in df.reset_index().iterrows():
    folium.Marker(
        [row['Latitude'], row['Longitude']],
        popup=f"{row['Name']} · Pop: {row['Population']:,}",
        icon=folium.Icon(color=tier_color(row['City_Tier']), icon='city', prefix='fa')
    ).add_to(cluster)

folium.LayerControl().add_to(m_cl)
m_cl.save('india_cluster_map.html')
display(m_cl)

---
# 📐 Statistical Analysis


## Population Distribution by City Tier (Box Plot)

In [ ]:
import seaborn as sns

order = ['Tier 1 (Metro)', 'Tier 1', 'Tier 2', 'Tier 3']
palette = {'Tier 1 (Metro)': '#c0392b', 'Tier 1': '#e67e22',
           'Tier 2': '#2980b9', 'Tier 3': '#27ae60'}

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df.reset_index(),
    x='City_Tier', y='Population',
    order=order, palette=palette
)
plt.xlabel('City Tier', fontsize=12)
plt.ylabel('Population', fontsize=12)
plt.title('Population Distribution by City Tier', fontsize=14, fontweight='bold')
plt.xticks(rotation=20)
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x/1e6)}M' if x >= 1e6 else f'{int(x/1e3)}K')
)
plt.tight_layout()
plt.show()

## Elevation Distribution by Region (Violin Plot)

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(
    data=df.reset_index(),
    x='Region', y='Digital Elevation Model',
    palette='muted'
)
plt.xlabel('Region', fontsize=12)
plt.ylabel('Elevation (m)', fontsize=12)
plt.title('Elevation Distribution by Region (Violin Plot)', fontsize=14, fontweight='bold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Correlation Heatmap – Numeric Variables (Seaborn)

In [ ]:
num_cols = ['Latitude', 'Longitude', 'Population',
            'Pop_Density_km2', 'Digital Elevation Model']

corr = df.reset_index()[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    square=True, linewidths=0.5
)
plt.title('Correlation Heatmap – Numeric Columns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Population Histogram + KDE by Region (Seaborn)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
axes = axes.flatten()
regions = df.reset_index()['Region'].unique()

for i, region in enumerate(regions[:6]):
    subset = df.reset_index()[df.reset_index()['Region'] == region]['Population']
    if len(subset) > 0:
        axes[i].hist(subset / 1e6, bins=8, color='steelblue',
                     edgecolor='black', alpha=0.8)
        axes[i].set_title(region, fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Population (M)')
        axes[i].set_ylabel('Count')

plt.suptitle('Population Distribution by Region', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📊 Grouped / Comparative Analysis


## Top 5 Cities per Region

In [ ]:
import matplotlib.gridspec as gridspec

regions = df.reset_index()['Region'].unique()
n_regions = len(regions)
cols = 3
rows = (n_regions + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, region in enumerate(sorted(regions)):
    top5 = (
        df.reset_index()[df.reset_index()['Region'] == region]
        .nlargest(5, 'Population')
    )
    axes[i].barh(top5['Name'], top5['Population'] / 1e6, color='tomato', alpha=0.85)
    axes[i].set_title(f'{region}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Population (M)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Top 5 Cities by Population – per Region',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## GroupBy Region – Multiple Metrics

In [ ]:
grouped = df.reset_index().groupby('Region').agg(
    Num_Cities=('Name', 'count'),
    Total_Pop=('Population', 'sum'),
    Avg_Pop=('Population', 'mean'),
    Avg_Elev=('Digital Elevation Model', 'mean'),
    Avg_Density=('Pop_Density_km2', 'mean')
).round(0).astype(int)

grouped.columns = ['# Cities', 'Total Pop', 'Avg Pop',
                    'Avg Elev (m)', 'Avg Density (/km²)']
display(grouped.sort_values('Total Pop', ascending=False))

## Average Population Density by State (Top 15)

In [ ]:
state_density = (
    df.reset_index()
    .groupby('State')['Pop_Density_km2']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
plt.bar(state_density.index, state_density.values,
        color='slateblue', alpha=0.85, edgecolor='black')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.xlabel('State', fontsize=12)
plt.ylabel('Avg Density (people / km²)', fontsize=12)
plt.title('Average Population Density by State – Top 15',
          fontsize=14, fontweight='bold')
for i, v in enumerate(state_density.values):
    plt.text(i, v + 500, f'{int(v):,}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

---
# 🗺️ Interactive Choropleth Map


## Average Elevation by Region (Plotly Bar Choropleth)

In [ ]:
import plotly.express as px

avg_elev = (
    df.reset_index()
    .groupby('Region')['Digital Elevation Model']
    .mean()
    .reset_index()
    .round(1)
)
avg_elev.columns = ['Region', 'Avg Elevation (m)']

fig = px.bar(
    avg_elev.sort_values('Avg Elevation (m)', ascending=True),
    x='Avg Elevation (m)', y='Region',
    orientation='h',
    color='Avg Elevation (m)', color_continuous_scale='Viridis',
    title=' Average City Elevation by Region (Interactive)',
    text='Avg Elevation (m)'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

## Population vs Elevation by Tier (Animated Scatter)

In [ ]:
fig = px.scatter(
    df.reset_index(),
    x='Longitude', y='Latitude',
    size='Population', color='Digital Elevation Model',
    animation_frame='City_Tier',
    hover_name='Name',
    color_continuous_scale='Plasma',
    size_max=40,
    title='🎞️ Animated: Cities by Tier – Lon / Lat / Population / Elevation',
    labels={'Longitude':'Longitude (°E)', 'Latitude':'Latitude (°N)'}
)
fig.update_layout(height=600)
fig.show()

## Sunburst – Region → Tier → City (Population)

In [ ]:
fig = px.sunburst(
    df.reset_index(),
    path=['Region', 'City_Tier', 'Name'],
    values='Population',
    color='Population',
    color_continuous_scale='RdYlGn',
    title='☀️ Population Sunburst – Region → Tier → City'
)
fig.update_layout(height=650)
fig.show()

---
# 🔬 Scatter Plot Matrix (Seaborn Pair Plot)


In [ ]:
pair_cols = ['Population', 'Pop_Density_km2',
             'Digital Elevation Model', 'Latitude']

g = sns.pairplot(
    df.reset_index()[pair_cols + ['City_Tier']],
    hue='City_Tier',
    vars=pair_cols,
    palette={
        'Tier 1 (Metro)':'#c0392b',
        'Tier 1':'#e67e22',
        'Tier 2':'#2980b9',
        'Tier 3':'#27ae60'
    },
    diag_kind='kde',
    plot_kws={'alpha': 0.6}
)
g.figure.suptitle('Scatter Plot Matrix – Key City Variables',
                  y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🤖 K-Means Geographic Clustering


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_r = df.reset_index()

# Features: Latitude, Longitude, Population, Elevation
features = df_r[['Latitude','Longitude','Population','Digital Elevation Model']].dropna()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Fit K-Means with 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_r.loc[features.index, 'Cluster'] = kmeans.fit_predict(X_scaled)
df_r['Cluster'] = df_r['Cluster'].fillna(-1).astype(int).astype(str)

# Plot clusters on a map
colors = {'0':'#e74c3c','1':'#3498db','2':'#2ecc71','3':'#f39c12','4':'#9b59b6'}

plt.figure(figsize=(12, 10))
for cluster, grp in df_r.groupby('Cluster'):
    plt.scatter(grp['Longitude'], grp['Latitude'],
                s=grp['Population']/80000, alpha=0.8,
                c=colors.get(cluster,'gray'),
                edgecolors='black', linewidths=0.4,
                label=f'Cluster {cluster}')
    for _, row in grp[grp['Population']>3_000_000].iterrows():
        plt.text(row['Longitude']+0.3, row['Latitude'],
                 row['Name'], fontsize=8, fontweight='bold')

plt.legend(title='K-Means Cluster', fontsize=9)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.title('K-Means Geographic Clustering of Indian Cities (k=5)',
          fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## K-Means Clusters – Interactive (Plotly)

In [ ]:
fig = px.scatter_geo(
    df_r.dropna(subset=['Cluster']),
    lat='Latitude', lon='Longitude',
    color='Cluster',
    size='Population',
    hover_name='Name',
    hover_data={'State':True,'Population':True,
                'Digital Elevation Model':True,'Cluster':True},
    size_max=35,
    title='🤖 K-Means Clusters – Interactive India City Map',
    center=dict(lat=22, lon=80),
    projection='equirectangular',
    scope='asia'
)
fig.update_geos(
    projection_scale=4,
    showocean=True, oceancolor='lightblue',
    showland=True,  landcolor='lightyellow',
    showcountries=True
)
fig.update_layout(height=600)
fig.show()

---
# 📈 Linear Regression – Elevation vs Population


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

df_lr = df.reset_index().dropna(subset=['Digital Elevation Model','Population'])
X = df_lr[['Digital Elevation Model']].values
y = df_lr['Population'].values

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

plt.figure(figsize=(10, 6))
plt.scatter(X, y/1e6, alpha=0.7, s=60, c='steelblue',
            edgecolors='black', linewidths=0.4, label='Cities')
plt.plot(sorted(X), sorted(y_pred/1e6), color='red',
         linewidth=2, linestyle='--', label='Regression Line')

for _, row in df_lr[df_lr['Population']>4_000_000].iterrows():
    plt.annotate(row['Name'],
                 (row['Digital Elevation Model'], row['Population']/1e6),
                 fontsize=8, ha='left', color='darkred')

r2 = model.score(X, y)
plt.xlabel('Elevation (m)', fontsize=12)
plt.ylabel('Population (Millions)', fontsize=12)
plt.title(f'Linear Regression: Elevation vs Population  (R² = {r2:.3f})',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout(); plt.show()


---
# 📉 Cumulative Population Distribution (CDF)


In [ ]:
sorted_pop = df.reset_index().sort_values('Population')['Population']
cdf = sorted_pop.rank(method='max').div(len(sorted_pop))

plt.figure(figsize=(10, 5))
plt.plot(sorted_pop/1e6, cdf, color='steelblue', linewidth=2)
plt.fill_between(sorted_pop/1e6, cdf, alpha=0.15, color='steelblue')
plt.axvline(sorted_pop.median()/1e6, color='red', linestyle='--',
            linewidth=1.5, label=f'Median: {sorted_pop.median()/1e6:.2f}M')
plt.axvline(sorted_pop.mean()/1e6, color='orange', linestyle='--',
            linewidth=1.5, label=f'Mean: {sorted_pop.mean()/1e6:.2f}M')
plt.xlabel('Population (Millions)', fontsize=12)
plt.ylabel('Cumulative Proportion', fontsize=12)
plt.title('Cumulative Distribution of Indian City Populations',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout(); plt.show()

---
# 📊 Multi-Metric Comparison by City Tier


In [ ]:
tier_stats = df.reset_index().groupby('City_Tier').agg(
    Avg_Pop   =('Population',            'mean'),
    Avg_Elev  =('Digital Elevation Model','mean'),
    Avg_Dens  =('Pop_Density_km2',        'mean'),
    Num_Cities=('Name',                   'count')
).round(0).reset_index()

order = ['Tier 1 (Metro)', 'Tier 1', 'Tier 2', 'Tier 3']
tier_stats = tier_stats.set_index('City_Tier').reindex(order).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#c0392b','#e67e22','#2980b9','#27ae60']

axes[0].bar(tier_stats['City_Tier'], tier_stats['Avg_Pop']/1e6,
            color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Avg Population (M)', fontweight='bold')
axes[0].set_xticklabels(tier_stats['City_Tier'], rotation=20, ha='right')

axes[1].bar(tier_stats['City_Tier'], tier_stats['Avg_Elev'],
            color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Avg Elevation (m)', fontweight='bold')
axes[1].set_xticklabels(tier_stats['City_Tier'], rotation=20, ha='right')

axes[2].bar(tier_stats['City_Tier'], tier_stats['Avg_Dens'],
            color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title('Avg Density (/km²)', fontweight='bold')
axes[2].set_xticklabels(tier_stats['City_Tier'], rotation=20, ha='right')

plt.suptitle('City Tier Comparison – Population / Elevation / Density',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# 🏅 Top 3 Most Populated Cities per State


In [ ]:
top3_state = (
    df.reset_index()
    .sort_values('Population', ascending=False)
    .groupby('State')
    .head(3)
    [['State','Name','Population','City_Tier']]
    .reset_index(drop=True)
)
top3_state['Population_M'] = (top3_state['Population']/1e6).round(2)
display(top3_state.sort_values(['State','Population'],
                                ascending=[True,False]))

In [ ]:
ranked = (
    df.reset_index()
    .sort_values('Population', ascending=False)
    .reset_index(drop=True)
)
ranked.index = ranked.index + 1  # 1-based rank
ranked.index.name = 'Rank'

display(
    ranked[['Name','State','Region','Population',
            'Digital Elevation Model','City_Tier']]
    .rename(columns={'Digital Elevation Model':'Elev (m)'})
    .style
    .background_gradient(subset=['Population'], cmap='Reds')
    .background_gradient(subset=['Elev (m)'], cmap='terrain')
    .format({'Population': '{:,}', 'Elev (m)': '{:.0f}'})
)

---
# 🏛️ State-wise Population Analysis


## Total Population by State (Top 15)

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt

state_pop = (
    df.reset_index()
    .groupby('State')['Population']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

fig = px.bar(
    state_pop,
    x='State', y='Population',
    color='Population',
    color_continuous_scale='Reds',
    title='🏛️ Total Urban Population by State (Top 15)',
    text=state_pop['Population'].apply(lambda x: f'{x/1e6:.1f}M')
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-40, showlegend=False,
                  yaxis_tickformat=',')
fig.show()

## Number of Cities per State (Top 15)

In [ ]:
state_count = (
    df.reset_index()
    .groupby('State')['Name']
    .count()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
state_count.columns = ['State', 'City Count']

plt.figure(figsize=(12, 6))
plt.bar(state_count['State'], state_count['City Count'],
        color='steelblue', alpha=0.85, edgecolor='black')
plt.xticks(rotation=40, ha='right', fontsize=10)
plt.xlabel('State', fontsize=12)
plt.ylabel('Number of Cities', fontsize=12)
plt.title('Cities per State – Top 15', fontsize=14, fontweight='bold')
for i, v in enumerate(state_count['City Count']):
    plt.text(i, v + 0.05, str(v), ha='center', fontsize=10, fontweight='semibold')
plt.tight_layout()
plt.show()

---
# 🔬 Population Density vs Population (Interactive)


In [ ]:
fig = px.scatter(
    df.reset_index(),
    x='Pop_Density_km2', y='Population',
    color='City_Tier', size='Population',
    hover_name='Name',
    hover_data={'State':True, 'Region':True,
                'Pop_Density_km2':True, 'Population':True},
    log_x=True,
    title='🔬 Population Density vs Total Population (Log Scale)',
    labels={'Pop_Density_km2':'Density (people/km², log scale)',
            'Population':'Population'},
    color_discrete_map={
        'Tier 1 (Metro)':'#c0392b', 'Tier 1':'#e67e22',
        'Tier 2':'#2980b9', 'Tier 3':'#27ae60'
    }
)
fig.update_layout(height=550, yaxis_tickformat=',')
fig.show()

---
# ⛰️ Elevation Distribution by City Tier (Violin)


In [ ]:
import seaborn as sns

order = ['Tier 1 (Metro)', 'Tier 1', 'Tier 2', 'Tier 3']
palette = {
    'Tier 1 (Metro)':'#c0392b', 'Tier 1':'#e67e22',
    'Tier 2':'#2980b9', 'Tier 3':'#27ae60'
}

plt.figure(figsize=(11, 6))
sns.violinplot(
    data=df.reset_index(),
    x='City_Tier', y='Digital Elevation Model',
    order=order, palette=palette, inner='box'
)
plt.xlabel('City Tier', fontsize=12)
plt.ylabel('Elevation (m)', fontsize=12)
plt.title('Elevation Distribution by City Tier', fontsize=14, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---
# 🌡️ Population Density Weighted Heatmap (Folium)
> Brighter = Higher population density (people/km²)


In [ ]:
from folium.plugins import HeatMap

m_dens = folium.Map(
    location=[22.5, 80.0], zoom_start=5,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri WorldImagery'
)

heat_data = [
    [row['Latitude'], row['Longitude'], row['Pop_Density_km2'] / 10000]
    for _, row in df.reset_index().iterrows()
]

HeatMap(
    heat_data,
    radius=30, blur=20, min_opacity=0.3,
    gradient={'0.3':'navy','0.5':'blue','0.65':'cyan',
              '0.8':'yellow','1':'red'}
).add_to(m_dens)

m_dens.save('india_density_heatmap.html')
display(m_dens)

---
# 📊 Advanced Statistics – Full Breakdown by State


In [ ]:
state_full = (
    df.reset_index()
    .groupby('State')
    .agg(
        Num_Cities=('Name',                   'count'),
        Total_Pop =('Population',              'sum'),
        Max_Pop   =('Population',              'max'),
        Avg_Pop   =('Population',              'mean'),
        Avg_Elev  =('Digital Elevation Model', 'mean'),
        Avg_Dens  =('Pop_Density_km2',         'mean'),
        Region    =('Region',                  'first')
    )
    .round(0)
    .astype({'Num_Cities':int, 'Total_Pop':int,
             'Max_Pop':int, 'Avg_Pop':int,
             'Avg_Elev':int, 'Avg_Dens':int})
    .sort_values('Total_Pop', ascending=False)
    .reset_index()
)

display(
    state_full.style
    .background_gradient(subset=['Total_Pop'], cmap='Reds')
    .background_gradient(subset=['Avg_Elev'], cmap='terrain')
    .background_gradient(subset=['Avg_Dens'], cmap='YlOrRd')
    .format({
        'Total_Pop':'{:,}', 'Max_Pop':'{:,}',
        'Avg_Pop':'{:,}', 'Avg_Elev':'{:,}',
        'Avg_Dens':'{:,}'
    })
)

---
# 🥧 Population Share by Region & Tier


In [ ]:
fig = px.pie(
    df.reset_index(), names='Region', values='Population',
    title='🌍 Population Share by Geographic Region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.35
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

In [ ]:
fig = px.pie(
    df.reset_index(), names='City_Tier', values='Population',
    title='🏙️ Population Share by City Tier',
    color_discrete_map={
        'Tier 1 (Metro)':'#c0392b', 'Tier 1':'#e67e22',
        'Tier 2':'#2980b9',         'Tier 3':'#27ae60'
    },
    hole=0.35
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

---
# 💠 Bubble Map – Density × Elevation × Population (Plotly)


In [ ]:
fig = px.scatter(
    df.reset_index(),
    x='Digital Elevation Model',
    y='Pop_Density_km2',
    size='Population',
    color='Region',
    hover_name='Name',
    hover_data={'State':True, 'City_Tier':True,
                'Population':True,
                'Digital Elevation Model':True,
                'Pop_Density_km2':True},
    title='💠 Elevation vs Density – Bubble = Population',
    labels={
        'Digital Elevation Model':'Elevation (m)',
        'Pop_Density_km2':'Density (people/km²)'
    },
    size_max=50
)
fig.update_layout(height=600,
                  yaxis_tickformat=',',
                  xaxis_tickformat=',')
fig.show()

---
# 🗺️ Region-wise Scatter Map (Matplotlib)


In [ ]:
region_colors = {
    'North':'#e74c3c', 'South':'#3498db',
    'East':'#2ecc71',  'West':'#f39c12',
    'Central':'#9b59b6', 'Northeast':'#1abc9c', 'Other':'#95a5a6'
}

plt.figure(figsize=(13, 11))
for region, grp in df.reset_index().groupby('Region'):
    plt.scatter(
        grp['Longitude'], grp['Latitude'],
        s=grp['Population'] / 60000,
        c=region_colors.get(region, 'gray'),
        alpha=0.8, edgecolors='black',
        linewidths=0.4, label=region
    )
    for _, row in grp[grp['Population'] > 3_000_000].iterrows():
        plt.annotate(
            row['Name'],
            (row['Longitude'], row['Latitude']),
            fontsize=8, fontweight='bold',
            xytext=(5, 5), textcoords='offset points'
        )

plt.legend(title='Region', fontsize=9, loc='lower left')
plt.xlabel('Longitude (°E)', fontsize=12)
plt.ylabel('Latitude (°N)', fontsize=12)
plt.title('India Cities by Region – Bubble = Population',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📐 Parallel Coordinates Plot
> Compare Population, Density, Elevation, and Latitude across all cities at once.


In [ ]:
import plotly.graph_objects as go

df_pc = df.reset_index().dropna(
    subset=['Population','Pop_Density_km2',
            'Digital Elevation Model','Latitude']
)

tier_map = {'Tier 1 (Metro)':0, 'Tier 1':1, 'Tier 2':2, 'Tier 3':3}
df_pc['Tier_Num'] = df_pc['City_Tier'].map(tier_map)

fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=df_pc['Tier_Num'],
        colorscale=[[0,'#c0392b'],[0.33,'#e67e22'],
                    [0.66,'#2980b9'],[1,'#27ae60']],
        showscale=True
    ),
    dimensions=[
        dict(label='Population (M)',
             values=df_pc['Population']/1e6,
             range=[0, df_pc['Population'].max()/1e6]),
        dict(label='Density (/km²)',
             values=df_pc['Pop_Density_km2'],
             range=[0, df_pc['Pop_Density_km2'].max()]),
        dict(label='Elevation (m)',
             values=df_pc['Digital Elevation Model'],
             range=[0, df_pc['Digital Elevation Model'].max()]),
        dict(label='Latitude (°N)',
             values=df_pc['Latitude'],
             range=[df_pc['Latitude'].min(), df_pc['Latitude'].max()]),
        dict(label='Longitude (°E)',
             values=df_pc['Longitude'],
             range=[df_pc['Longitude'].min(), df_pc['Longitude'].max()]),
    ]
))
fig.update_layout(
    title='📐 Parallel Coordinates – All City Variables',
    height=500
)
fig.show()

---
# 📊 Population Bins – How Many Cities in Each Range?


In [ ]:
import numpy as np

bins   = [0, 500_000, 1_000_000, 2_000_000,
          5_000_000, 10_000_000, 25_000_000]
labels = ['<5L', '5L–10L', '10L–20L',
          '20L–50L', '50L–1Cr', '>1Cr']

df_r = df.reset_index()
df_r['Pop_Bin'] = pd.cut(df_r['Population'], bins=bins, labels=labels)
bin_counts = df_r['Pop_Bin'].value_counts().reindex(labels)

plt.figure(figsize=(10, 5))
bars = plt.bar(bin_counts.index, bin_counts.values,
               color=['#27ae60','#3498db','#9b59b6',
                      '#e67e22','#e74c3c','#c0392b'],
               edgecolor='black', alpha=0.88)
for bar, v in zip(bars, bin_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             str(v), ha='center', fontsize=11, fontweight='bold')
plt.xlabel('Population Range', fontsize=12)
plt.ylabel('Number of Cities', fontsize=12)
plt.title('How Many Indian Cities Fall in Each Population Range?',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📊 Stacked Bar – City Tier Breakdown per Region


In [ ]:
pivot = (
    df.reset_index()
    .groupby(['Region','City_Tier'])['Name']
    .count()
    .unstack(fill_value=0)
)
tier_order = ['Tier 1 (Metro)','Tier 1','Tier 2','Tier 3']
pivot = pivot.reindex(columns=tier_order, fill_value=0)

tier_colors = ['#c0392b','#e67e22','#2980b9','#27ae60']

pivot.plot(
    kind='bar', stacked=True,
    color=tier_colors, figsize=(12, 6),
    edgecolor='black', linewidth=0.4, alpha=0.88
)
plt.xlabel('Region', fontsize=12)
plt.ylabel('Number of Cities', fontsize=12)
plt.title('City Tier Breakdown per Region (Stacked Bar)',
          fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.legend(title='City Tier', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
# 🏔️ Highest vs Lowest Elevation Cities


In [ ]:
top5_high = df.reset_index().nlargest(5, 'Digital Elevation Model')[
    ['Name','State','Digital Elevation Model','Population','City_Tier']
]
top5_low  = df.reset_index().nsmallest(5, 'Digital Elevation Model')[
    ['Name','State','Digital Elevation Model','Population','City_Tier']
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top5_high['Name'], top5_high['Digital Elevation Model'],
             color='#8e44ad', edgecolor='black', alpha=0.85)
axes[0].set_title('🏔️ Top 5 Highest Cities', fontweight='bold')
axes[0].set_xlabel('Elevation (m)')
for i, (_, row) in enumerate(top5_high.iterrows()):
    axes[0].text(10, i, f"{row['Digital Elevation Model']:.0f} m",
                 va='center', fontsize=9, fontweight='bold', color='white')

axes[1].barh(top5_low['Name'], top5_low['Digital Elevation Model'],
             color='#16a085', edgecolor='black', alpha=0.85)
axes[1].set_title('🌊 Top 5 Lowest Cities', fontweight='bold')
axes[1].set_xlabel('Elevation (m)')
for i, (_, row) in enumerate(top5_low.iterrows()):
    axes[1].text(0.5, i, f"{row['Digital Elevation Model']:.0f} m",
                 va='center', fontsize=9, fontweight='bold', color='white')

plt.suptitle('Highest vs Lowest Elevation Indian Cities',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 🏘️ Most & Least Dense Cities


In [ ]:
top5_dense  = df.reset_index().nlargest(5, 'Pop_Density_km2')[
    ['Name','State','Pop_Density_km2','Population','City_Tier']
]
top5_sparse = df.reset_index().nsmallest(5, 'Pop_Density_km2')[
    ['Name','State','Pop_Density_km2','Population','City_Tier']
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top5_dense['Name'], top5_dense['Pop_Density_km2'],
             color='#e74c3c', edgecolor='black', alpha=0.85)
axes[0].set_title('🏘️ Top 5 Most Dense Cities', fontweight='bold')
axes[0].set_xlabel('Density (people/km²)')
for i, v in enumerate(top5_dense['Pop_Density_km2']):
    axes[0].text(500, i, f'{v:,}', va='center',
                 fontsize=9, fontweight='bold', color='white')

axes[1].barh(top5_sparse['Name'], top5_sparse['Pop_Density_km2'],
             color='#27ae60', edgecolor='black', alpha=0.85)
axes[1].set_title('🌿 Top 5 Least Dense Cities', fontweight='bold')
axes[1].set_xlabel('Density (people/km²)')
for i, v in enumerate(top5_sparse['Pop_Density_km2']):
    axes[1].text(10, i, f'{v:,}', va='center',
                 fontsize=9, fontweight='bold', color='white')

plt.suptitle('Most vs Least Dense Indian Cities',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📊 Facet Charts – Population per Region (Interactive)


In [ ]:
fig = px.bar(
    df.reset_index().sort_values('Population', ascending=False)
      .groupby('Region').head(5)
      .reset_index(drop=True),
    x='Name', y='Population',
    color='City_Tier', facet_col='Region',
    facet_col_wrap=3,
    color_discrete_map={
        'Tier 1 (Metro)':'#c0392b', 'Tier 1':'#e67e22',
        'Tier 2':'#2980b9',         'Tier 3':'#27ae60'
    },
    title='🏙️ Top 5 Cities per Region (Faceted Bar Chart)',
    labels={'Population':'Population','Name':'City'}
)
fig.update_xaxes(tickangle=40)
fig.update_layout(height=700, yaxis_tickformat=',')
fig.show()

In [ ]:
df_r = df.reset_index()
df_r.describe().T[['count','mean','std','min','max']].round(1)

In [ ]:
df_r = df.reset_index()
df_r.groupby('Region')[[
    'Population','Pop_Density_km2','Digital Elevation Model'
]].mean().round(1)

In [ ]:
# ── Satellite Imagery Map – Contextily + GeoPandas ──────────────
!pip install contextily --quiet
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df.reset_index(),
    geometry=gpd.points_from_xy(
        df.reset_index()['Longitude'],
        df.reset_index()['Latitude']
    ),
    crs='EPSG:4326'
)

# Convert to Web Mercator (required for satellite tiles)
gdf_web = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(14, 16))

gdf_web.plot(
    ax=ax,
    column='Population',
    cmap='OrRd',
    markersize=gdf_web['Population'] / 80000,
    alpha=0.85,
    legend=True,
    legend_kwds={'label': 'Population', 'shrink': 0.5}
)

# 🛰️ Add Esri satellite imagery as basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    zoom=5
)

ax.set_title(
    '🛰️ India Cities on Satellite Imagery',
    fontsize=15, fontweight='bold'
)
ax.set_axis_off()
plt.tight_layout()
plt.show()